# ⛰ Core Lab 6 — Inside the GPU
**A Billion Small Steps · Hardware for AI · Step 3 · Book sections 3.1–3.2 · Foothold lessons 3.1–3.2**

Read the book sections first, then run this notebook from top to bottom. Runs on a **GPU**: Runtime → Change runtime type → **T4 GPU**.

**What you should see:** peak FLOPS from the parts; FP16 several times faster than FP32; a diverging kernel about 2× slower.

| Part | What you do | Book |
|---|---|---|
| A | Work out the peak, then measure FP32 vs FP16 | 3.1 |
| B | Make a warp diverge, and time the damage | 3.2 |

⏱️ About 2 hours. 📘 Textbook: Hwu, Kirk & El Hajj Ch 4 · H&P §4.4. Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run these cells first

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

In [ ]:
# CuPy lets Python compile and run small CUDA C kernels. It comes pre-installed on Colab.
try:                                # "try": attempt this; if it fails, jump to "except" instead of crashing
    import cupy as cp               # CuPy: NumPy-like arrays on the GPU, plus our own CUDA kernels
    HAVE_CUPY = device == "cuda"    # we can only use it if there is also a GPU
    print("CuPy", cp.__version__, "ready" if HAVE_CUPY else "(installed, but no GPU in this session)")   # say whether it's usable
except ImportError:                 # CuPy is not installed
    HAVE_CUPY = False               # remember that, so later cells can say so
    print("CuPy not found. On Colab with a GPU, run:  !pip install cupy-cuda12x   then re-run this cell.")   # how to install it

def cuda_time_ms(launch, repeats=5):                # time GPU work with the GPU's own clock (CUDA events); returns ms per run
    launch(); cp.cuda.Device().synchronize()        # warm-up run, then wait for it to finish
    start, end = cp.cuda.Event(), cp.cuda.Event()   # two time-stamps on the GPU
    start.record()                                  # stamp "start" in the GPU's queue
    for _ in range(repeats):                        # several runs
        launch()                                    # queue the kernel once more
    end.record(); end.synchronize()                 # stamp "end", and wait until the GPU reaches it
    return cp.cuda.get_elapsed_time(start, end) / repeats   # time between the stamps ÷ runs

## Part A · Work Out the Peak, Then Measure  *(Section 3.1)*

Peak FLOPS = SMs × units per SM × FLOPs per unit per clock × clock rate. For the T4: 40 SMs; 64 FP32 cores and 8 Tensor Cores per SM; one FMA (2 FLOPs) per core per clock; 64 FMAs per Tensor Core per clock.

In [ ]:
# the T4's published design: 40 SMs, each with 64 FP32 cores and 8 Tensor Cores, boost clock 1.59 GHz
t4_sms, clock_ghz = 40, 1.59
# FP32: SMs × 64 cores × 2 FLOPs per fused multiply-add × clocks per second
fp32_peak = t4_sms * 64 * 2 * clock_ghz * 1e9
# FP16 Tensor Cores: SMs × 8 Tensor Cores × 64 multiply-adds each × 2 FLOPs × clocks per second
fp16_peak = t4_sms * 8 * 64 * 2 * clock_ghz * 1e9
print(f"T4 peak FP32 (CUDA cores)  : {fp32_peak / 1e12:5.1f} TFLOPS")   # ÷ 1e12 gives TFLOPS
print(f"T4 peak FP16 (Tensor Cores): {fp16_peak / 1e12:5.1f} TFLOPS")   # the Tensor Cores' peak
# the GPU in THIS session: how many SMs does it have?
sms = torch.cuda.get_device_properties(0).multi_processor_count
print(f"This GPU has {sms} SMs")   # show it

In [ ]:
N = 4096   # the matrices are 4096 × 4096
measured = {}   # number format → measured TFLOPS
# dtype is the number format: 32-bit floats, then 16-bit floats
for dtype in [torch.float32, torch.float16]:
    # two random 4096 × 4096 matrices on the GPU, in this format
    a = torch.rand(N, N, device=device, dtype=dtype); b = torch.rand(N, N, device=device, dtype=dtype)
    # average time of one multiply; speed = 2N³ FLOPs ÷ time, ÷ 1e12 gives TFLOPS
    t = timeit(lambda: a @ b, repeats=10)
    measured[dtype] = 2 * N**3 / t / 1e12   # speed = 2N³ FLOPs ÷ time, in TFLOPS
    print(f"{str(dtype):15s}: {t*1e3:7.2f} ms  →  {measured[dtype]:5.1f} TFLOPS measured")   # show the time and the speed
print(f"FP16 is {measured[torch.float16] / measured[torch.float32]:.1f}× faster than FP32 on this GPU")   # compare the two formats

### ✏️ YOUR TURN (A)
What percentage of the T4's computed peak did each precision reach? Why is FP16 so much faster? (Hint: which units does it use?)

*Your answer:*

## Part B · Branch Divergence  *(Section 3.2)*

Below is a real CUDA kernel. Every thread takes one of two expensive paths.
- `mode = 0`: the path is chosen by `(i / 32) % 2`, so all 32 threads in a warp agree.
- `mode = 1`: the path is chosen by `i % 2`, so every warp splits in half.

Same threads, same total work. Only the arrangement differs.

In [ ]:
# A CUDA C kernel, kept in a Python string. In C, comments start with // instead of #.
branchy_src = r'''
// __global__ marks a kernel: code that runs on the GPU and is launched from the CPU
extern "C" __global__
void branchy(float *x, int n, int mode, int work) {   // x = the numbers, n = how many, mode = 0 or 1, work = loop steps
    // this thread's element; spare threads past the end stop at once
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;   // spare threads past the end stop here
    // mode 0: the path depends on (i / 32) % 2, so all 32 threads of a warp agree
    // mode 1: the path depends on i % 2, so neighbouring threads disagree
    int path = (mode == 0) ? ((i / 32) % 2) : (i % 2);
    float v = x[i];   // read this thread's number
    // two equally expensive paths, A and B
    if (path) { for (int k = 0; k < work; k++) v = v * 0.9999f + 0.0001f; }
    else      { for (int k = 0; k < work; k++) v = v * 1.0001f - 0.0001f; }   // (path B, just as long as path A)
    x[i] = v;   // write the result back
}
'''
# CuPy compiles the C code into a GPU kernel
branchy = cp.RawKernel(branchy_src, "branchy")
# 1 << 22 = 2²² ≈ 42 lakh threads, each doing 2,000 loop steps; an array of 1.0s in GPU memory
n, work = 1 << 22, 2000
x = cp.ones(n, dtype=cp.float32)   # an array of 1.0s in GPU memory
# 256 threads per block, and enough blocks to cover n (rounded up)
threads = 256; blocks = (n + threads - 1) // threads
times = {}   # mode → time in ms
for mode, name in [(0, "warps agree   (i/32)%2"), (1, "warps diverge  i%2   ")]:   # run both arrangements
    # launch: kernel((blocks,), (threads per block,), (the kernel's arguments)); cp.int32 is a C "int"
    times[mode] = cuda_time_ms(lambda: branchy((blocks,), (threads,), (x, cp.int32(n), cp.int32(mode), cp.int32(work))))
    print(f"{name}: {times[mode]:7.2f} ms")   # show this arrangement's time
print(f"Divergence cost: {times[1] / times[0]:.2f}× slower")   # how much slower the diverging version was

### ✏️ YOUR TURN (B)
1. Was the diverging version about 2× slower? Explain why using the word *warp*.
2. Change `(i / 32) % 2` to `(i / 16) % 2` and re-run. Predict first: does it diverge?

*Your answers:*

---
## 🎓 Wrap-up
- A GPU's peak = SMs × cores per SM × 2 FLOPs × clock; Tensor Cores in FP16 go far higher.
- 32 threads (a warp) share one instruction; if they split on an if/else, both paths run one after the other.
- **Next — Core Lab 7: Feeding the GPU.** Hiding memory latency with threads, and coalesced vs strided reads.